In [1]:
# install libraries
!pip install pandas matplotlib nltk webvtt-py

In [2]:
# Download NLTK
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab') # Added punkt_tab download

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [3]:
# Imports
import re
import webvtt
import pandas as pd
from webvtt import WebVTT
from datetime import timedelta
from collections import Counter

from nltk import bigrams
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer, PorterStemmer

In [4]:
# Upload your files
from google.colab import files
uploaded = files.upload()

Saving all_comments.txt to all_comments.txt
Saving captions.en.vtt to captions.en.vtt


In [5]:
# Check Exiting Files in Directory
!ls

all_comments.txt  captions.en.vtt  sample_data


In [6]:
# Comment Parser
def structure_comments_from_txt(filepath='all_comments.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    comments_data = []

    # Each line looks like:
    # "1. @username [2025-10-19T22:03:01Z] 0 Likes → This was good!"
    pattern = re.compile(
        r'^\s*\d+\.\s*@([\w\-\_]+)\s*\[([0-9T:\-Z]+)\]\s*(\d+)\s*Likes?\s*→\s*(.*)$'
    )

    for line in lines:
        line = line.strip()
        match = pattern.match(line)
        if match:
            username = match.group(1)
            timestamp = match.group(2)
            likes = int(match.group(3))
            comment_text = match.group(4).strip()

            # Add length-based features
            char_length = len(comment_text)
            word_count = len(comment_text.split())
            avg_word_length = round(char_length / word_count, 2) if word_count else 0

            comments_data.append({
                'username': username,
                'timestamp_text': timestamp,
                'likes': likes,
                'comment_text': comment_text,
                'char_length': char_length,
                'word_count': word_count,
                'avg_word_length': avg_word_length
            })

    return pd.DataFrame(comments_data)

comments_df = structure_comments_from_txt()
comments_df.head(20)

,username,timestamp_text,likes,comment_text,char_length,word_count,avg_word_length
0,LafonClark-r3r,2026-01-24T17:41:22Z,0,Post Quantum Polymorphic Prompt Injection Via ...,123,10,12.30
1,MahboobaliAli-i5l,2026-01-23T14:29:07Z,0,"Absolute perfection, what a tutorial!",37,5,7.40
2,Blackwhite2277,2026-01-22T21:27:44Z,0,"If we are allowing AI to do EVERYTHING, only a...",204,40,5.10
3,stamdar1,2026-01-22T17:27:42Z,0,honestly so impressed by him writing backwards,46,7,6.57
4,DhivyabharathiB-t3v,2026-01-22T04:26:38Z,0,Informative.. Clear Info sir.,29,4,7.25
5,marpro212,2026-01-20T12:51:05Z,0,I’m impressed with how you can write in a mirr...,83,14,5.93
6,nestorrivas7703,2026-01-20T11:38:45Z,0,NHIs seem to be the next generation of service...,58,11,5.27
7,cumMan270,2026-01-19T06:46:35Z,0,Your explanation of shadow AI is a good exampl...,69,14,4.93
8,stantkatchenko1341,2026-01-19T04:42:15Z,0,"Wish list, let&#39;s talk about AI and AI secu...",148,28,5.29
9,m_0_h_1_t,2026-01-18T17:09:25Z,0,Thank you Jeff. I am a senior SOC Analyst and ...,191,38,5.03


In [7]:
# Comment Parser
def structure_comments_from_txt(filepath='all_comments.txt'):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    comments_data = []

    # Each line looks like:
    # "1. @username [2025-10-19T22:03:01Z] 0 Likes → This was good!"
    pattern = re.compile(
        r'^\s*\d+\.\s*@([\w\-\_]+)\s*\[([0-9T:\-Z]+)\]\s*(\d+)\s*Likes?\s*→\s*(.*)$'
    )

    for line in lines:
        line = line.strip()
        match = pattern.match(line)
        if match:
            username = match.group(1)
            timestamp = match.group(2)
            likes = int(match.group(3))
            comment_text = match.group(4).strip()

            # Add length-based features
            char_length = len(comment_text)
            word_count = len(comment_text.split())
            avg_word_length = round(char_length / word_count, 2) if word_count else 0

            comments_data.append({
                'username': username,
                'timestamp_text': timestamp,
                'likes': likes,
                'comment_text': comment_text,
                'char_length': char_length,
                'word_count': word_count,
                'avg_word_length': avg_word_length
            })

    return pd.DataFrame(comments_data)

comments_df = structure_comments_from_txt()
comments_df.head(20)

,username,timestamp_text,likes,comment_text,char_length,word_count,avg_word_length
0,LafonClark-r3r,2026-01-24T17:41:22Z,0,Post Quantum Polymorphic Prompt Injection Via ...,123,10,12.30
1,MahboobaliAli-i5l,2026-01-23T14:29:07Z,0,"Absolute perfection, what a tutorial!",37,5,7.40
2,Blackwhite2277,2026-01-22T21:27:44Z,0,"If we are allowing AI to do EVERYTHING, only a...",204,40,5.10
3,stamdar1,2026-01-22T17:27:42Z,0,honestly so impressed by him writing backwards,46,7,6.57
4,DhivyabharathiB-t3v,2026-01-22T04:26:38Z,0,Informative.. Clear Info sir.,29,4,7.25
5,marpro212,2026-01-20T12:51:05Z,0,I’m impressed with how you can write in a mirr...,83,14,5.93
6,nestorrivas7703,2026-01-20T11:38:45Z,0,NHIs seem to be the next generation of service...,58,11,5.27
7,cumMan270,2026-01-19T06:46:35Z,0,Your explanation of shadow AI is a good exampl...,69,14,4.93
8,stantkatchenko1341,2026-01-19T04:42:15Z,0,"Wish list, let&#39;s talk about AI and AI secu...",148,28,5.29
9,m_0_h_1_t,2026-01-18T17:09:25Z,0,Thank you Jeff. I am a senior SOC Analyst and ...,191,38,5.03


In [10]:
# Cleaning pipeline
stop_words = set(stopwords.words('english'))
lemmatizer, stemmer = WordNetLemmatizer(), PorterStemmer()

def normalize_text(text: str) -> str:
    if text is None: return ""
    text = text.lower()
    text = re.sub(r'\[.*?]', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)      # change to [^a-z0-9\s] to keep digits
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def tokenize_text(text: str):
    if not text: return []
    return word_tokenize(text)

def remove_stopwords(tokens):
    return [w for w in tokens if w not in stop_words and len(w) > 2]

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(w) for w in tokens]

def stem_tokens(tokens):
    return [stemmer.stem(w) for w in tokens]

def clean_text_pipeline(text: str, use_lemmatization: bool = True):
    x = normalize_text(text)
    toks = tokenize_text(x)
    toks = remove_stopwords(toks)
    return lemmatize_tokens(toks) if use_lemmatization else stem_tokens(toks)

# Comments parser
def structure_comments_from_txt(filepath='all_comments.txt'):
    pat = re.compile(
        r'^\s*\d+\.\s*@([\w\-]+)\s*\[([0-9T:\-Z]+)\]\s*([\d,]+)\s*Likes?\s*\u2192\s*(.+)$'
    )
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for raw in f:
            m = pat.match(raw.strip())
            if not m:
                continue
            username  = m.group(1)
            timestamp = m.group(2)
            likes = int(m.group(3).replace(',', ''))
            comment = m.group(4).strip()
            rows.append({'username': username,
                         'timestamp': timestamp,
                         'likes': likes,
                         'comment_text': comment})
    return pd.DataFrame(rows, columns=['username','timestamp','likes','comment_text'])

# Captions parser
_SENT_SPLIT = re.compile(r'(?<=[.!?])\s+')

def _c(s: str) -> str:
    # minimal clean: strip, collapse spaces, drop stray quotes
    s = ' '.join(s.strip().split())
    # Corrected the string literal to include all desired quote characters for stripping
    return s.strip('"\'“”‘’')

def _deoverlap(prev_line: str, curr_line: str, k: int = 6) -> str:
    """
    Remove up to k-word overlap between the end of prev_line and start of curr_line.
    This kills the rolling-window repetition common in auto-captions.
    """
    a = prev_line.split()
    b = curr_line.split()
    for n in range(min(k, len(a), len(b)), 0, -1):
        if a[-n:] == b[:n]:
            return ' '.join(b[n:])
    return curr_line

def structure_captions_from_vtt(filepath='captions.en.vtt'):
    # read all cues (assumes webvtt works; no extra error handling)
    cues = [_c(cue.text) for cue in WebVTT().read(filepath) if cue.text.strip()]
    if not cues:
        return pd.DataFrame({'caption_sentence': []})

    # merge cues while removing small overlaps
    merged = [cues[0]]
    for t in cues[1:]:
        extra = _deoverlap(merged[-1], t, k=6)
        if extra:
            merged[-1] = merged[-1] + ' ' + extra

    # split into sentences
    big_text = ' '.join(merged)
    sentences = [s.strip() for s in _SENT_SPLIT.split(big_text) if s.strip()]

    return pd.DataFrame({'caption_sentence': sentences})

# Preview
comments_df = structure_comments_from_txt('all_comments.txt')
comments_df['cleaned_tokens'] = comments_df['comment_text'].apply(clean_text_pipeline)
print("Comments head:")
print(comments_df.head())

captions_df = structure_captions_from_vtt('captions.en.vtt')
captions_df['cleaned_tokens'] = captions_df['caption_sentence'].apply(clean_text_pipeline)
print("\nCaptions head:")
print(captions_df.head(20))

Comments head:
              username             timestamp  likes  \
0       LafonClark-r3r  2026-01-24T17:41:22Z      0   
1    MahboobaliAli-i5l  2026-01-23T14:29:07Z      0   
2       Blackwhite2277  2026-01-22T21:27:44Z      0   
3             stamdar1  2026-01-22T17:27:42Z      0   
4  DhivyabharathiB-t3v  2026-01-22T04:26:38Z      0   

                                        comment_text  \
0  Post Quantum Polymorphic Prompt Injection Via ...   
1              Absolute perfection, what a tutorial!   
2  If we are allowing AI to do EVERYTHING, only a...   
3     honestly so impressed by him writing backwards   
4                      Informative.. Clear Info sir.   

                                      cleaned_tokens  
0  [post, quantum, polymorphic, prompt, injection...  
1                   [absolute, perfection, tutorial]  
2  [allowing, everything, elite, group, people, a...  
3          [honestly, impressed, writing, backwards]  
4                    [informative, clear, 

In [11]:
# RESULTS & DISCUSSION

STOP = set(stopwords.words('english'))

def _count_sw(text):
    if pd.isna(text): return 0
    toks = word_tokenize(str(text).lower().strip())
    return sum(1 for t in toks if t in STOP)

def _top_tokens(df, col='cleaned_tokens', k=10):
    bag = Counter()
    for toks in df.get(col, []):
        if isinstance(toks, list): bag.update(toks)
    return bag.most_common(k)

def _issues(df, col):
    empty = (df[col].fillna("").str.strip() == "").sum()
    nonascii = df[col].fillna("").map(lambda s: any(ord(c)>127 for c in str(s))).sum()
    symbols = df[col].fillna("").map(
        lambda s: (len(str(s)) - sum(ch.isalnum() or ch.isspace() or ch in ".,!?'" for ch in str(s))) >= 3
    ).sum()
    return empty, nonascii, symbols

print("\n=== RESULTS & DISCUSSION ===")

# 1) Sizes
print(f"- Rows → comments: {comments_df.shape[0]} | captions: {captions_df.shape[0]}")

# 2) Before→After examples (2 each)
def _norm(s): return str(s).lower().strip().replace("\n"," ")
print("\nExamples — Comments (before → tokens)")
for _, r in comments_df[['comment_text','cleaned_tokens']].head(2).iterrows():
    print("•", _norm(r['comment_text'])[:120])
    print("  →", r['cleaned_tokens'][:12])

print("\nExamples — Captions (before → tokens)")
for _, r in captions_df[['caption_sentence','cleaned_tokens']].head(2).iterrows():
    print("•", _norm(r['caption_sentence'])[:120])
    print("  →", r['cleaned_tokens'][:12])

# 3) Exact stopword totals (report-ready single line)
com_sw = comments_df['comment_text'].apply(_count_sw).sum() if 'comment_text' in comments_df else 0
cap_sw = captions_df['caption_sentence'].apply(_count_sw).sum() if 'caption_sentence' in captions_df else 0
print(f"\nStopwords removed (exact count on originals): total={int(com_sw+cap_sw)} "
      f"(comments={int(com_sw)}, captions={int(cap_sw)})")

# 4) Top tokens (10)
print("\nTop tokens — Comments (10)")
print(_top_tokens(comments_df, 'cleaned_tokens', 10))
print("Top tokens — Captions (10)")
print(_top_tokens(captions_df, 'cleaned_tokens', 10))

# 5) Input issues (counts only)
if 'comment_text' in comments_df:
    e,n,s = _issues(comments_df,'comment_text')
    print(f"\nData issues — Comments: empty={e}, non_ascii={n}, symbols={s}")
if 'caption_sentence' in captions_df:
    e,n,s = _issues(captions_df,'caption_sentence')
    print(f"Data issues — Captions: empty={e}, non_ascii={n}, symbols={s}")

# 6) Engagement (only if likes available)
if 'likes' in comments_df:
    comments_df['tok_len']  = comments_df['cleaned_tokens'].map(lambda x: len(x) if isinstance(x,list) else 0)
    comments_df['char_len'] = comments_df['comment_text'].astype(str).str.len()
    comments_df['word_len'] = comments_df['comment_text'].astype(str).str.split().map(len)
    print("\nEngagement — correlations with likes")
    for c in ['word_len','tok_len','char_len']:
        print(f"  corr(likes, {c}) = {comments_df['likes'].corr(comments_df[c]):.3f}")
    print("\nTop 3 most-liked comments (user | likes | snippet)")
    for _, r in comments_df.sort_values('likes', ascending=False).head(3).iterrows():
        user = r['username'] if 'username' in comments_df else 'user'
        snippet = str(r['comment_text']).replace("\n"," ")[:80]
        print(f"  • {user} | {r['likes']} | {snippet}")


=== RESULTS & DISCUSSION ===
- Rows → comments: 216 | captions: 574

Examples — Comments (before → tokens)
• post quantum polymorphic prompt injection via deep fake <a href="http://www.youtube.com/results?search_query=%23ai">#ai<
  → ['post', 'quantum', 'polymorphic', 'prompt', 'injection', 'via', 'deep', 'fake', 'href', 'http', 'www', 'youtube']
• absolute perfection, what a tutorial!
  → ['absolute', 'perfection', 'tutorial']

Examples — Captions (before → tokens)
• it's become something of a tradition here on the ibm technology channel to do here on the ibm technology channel to do h
  → ['become', 'something', 'tradition', 'ibm', 'technology', 'channel', 'ibm', 'technology', 'channel', 'ibm', 'technology', 'channel']
• we did this in 2023, for the future.
  → ['future']

Stopwords removed (exact count on originals): total=7352 (comments=1978, captions=5374)

Top tokens — Comments (10)
[('video', 27), ('quantum', 26), ('shadow', 18), ('like', 18), ('tech', 16), ('deepfakes', 16), (

In [12]:
# Save cleaned data
comments_df.to_csv('cleaned_comments.csv', index=False)
captions_df.to_csv('cleaned_captions.csv', index=False)

In [13]:
df1 = pd.read_csv('cleaned_comments.csv')
df2 = pd.read_csv('cleaned_captions.csv')

In [14]:
print(df1.head(20))



                username             timestamp  likes  \
0         LafonClark-r3r  2026-01-24T17:41:22Z      0   
1      MahboobaliAli-i5l  2026-01-23T14:29:07Z      0   
2         Blackwhite2277  2026-01-22T21:27:44Z      0   
3               stamdar1  2026-01-22T17:27:42Z      0   
4    DhivyabharathiB-t3v  2026-01-22T04:26:38Z      0   
5              marpro212  2026-01-20T12:51:05Z      0   
6        nestorrivas7703  2026-01-20T11:38:45Z      0   
7              cumMan270  2026-01-19T06:46:35Z      0   
8     stantkatchenko1341  2026-01-19T04:42:15Z      0   
9              m_0_h_1_t  2026-01-18T17:09:25Z      0   
10         youtube-64bts  2026-01-17T06:17:37Z      0   
11    RajatThakur_Canada  2026-01-16T14:59:14Z      0   
12         RedwanBakkass  2026-01-16T08:44:13Z      0   
13  estudiostecnicos2001  2026-01-15T20:13:54Z      0   
14            AuliTakala  2026-01-13T23:51:07Z      2   
15     lorenzococcia8665  2026-01-13T12:33:25Z      2   
16           EGORbodrov1  2026-

In [15]:
print(df2.head(200))

                                      caption_sentence  \
0    It's become something of a tradition here on t...   
1                 We did this in 2023, for the future.   
2                 We did this in 2023, for the future.   
3     We did this in 2023, again in 2024, and in 2025.   
4             And I'm back again in 2024, and in 2025.   
..                                                 ...   
195  So, I'm not missing on agents two years in a n...   
196                           I knew they were coming.   
197                                        I just row.   
198                           I knew they were coming.   
199                                        I just row.   

                                        cleaned_tokens  
0    ['become', 'something', 'tradition', 'ibm', 't...  
1                                           ['future']  
2                                           ['future']  
3                                                   []  
4                 